# Análise Exploratória — Intraday Regression Bands

Notebook de exploração dos dados e indicadores antes de rodar os backtests formais.

In [ ]:
import sys
from pathlib import Path

# Adiciona raiz do projeto ao path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from src.config import load_config, get_project_root
from src.indicators import (
    log_return, rsi, stochastic, bollinger_bands,
    regression_bands, return_residual_bands,
    technical_confluence_score, compute_all_features
)

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

cfg = load_config()
root = get_project_root()
print('Configuração carregada.')
print('Ativos:', cfg['data']['symbols'])
print('Intervalos:', cfg['data']['intervals'])

## 1. Carregando dados processados

In [ ]:
symbol = 'BTCUSDT'
interval = '1h'  # Ajuste conforme disponível

data_path = root / 'data' / 'processed' / f'{symbol}_{interval}.parquet'

if data_path.exists():
    df = pd.read_parquet(data_path)
    print(f'Shape: {df.shape}')
    print(f'Período: {df["timestamp"].min()} → {df["timestamp"].max()}')
    df.head(3)
else:
    print(f'Arquivo não encontrado: {data_path}')
    print('Execute: python scripts/01_download_data.py && python scripts/02_prepare_data.py')

## 2. Visualização de preços e volume

In [ ]:
if 'df' in dir() and not df.empty:
    # Usar amostra recente para visualização
    sample = df.tail(500).copy()
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    
    axes[0].plot(sample['timestamp'], sample['close'], linewidth=0.8, color='#2c3e50')
    axes[0].set_title(f'{symbol} {interval} — Close Price', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Preço (USD)')
    axes[0].spines['top'].set_visible(False)
    axes[0].spines['right'].set_visible(False)
    
    axes[1].bar(sample['timestamp'], sample['volume'], width=0.001, color='#3498db', alpha=0.6)
    axes[1].set_title('Volume', fontsize=11)
    axes[1].set_ylabel('Volume')
    axes[1].set_xlabel('Data')
    axes[1].spines['top'].set_visible(False)
    axes[1].spines['right'].set_visible(False)
    
    fig.tight_layout()
    plt.show()

## 3. Indicadores técnicos

In [ ]:
if 'df' in dir() and not df.empty:
    sample = df.tail(1000).copy().reset_index(drop=True)
    close = sample['close']
    high = sample['high']
    low = sample['low']
    
    # Bollinger Bands
    bb_upper, bb_mid, bb_lower = bollinger_bands(close, window=20, k=2.0)
    
    # RSI
    rsi_14 = rsi(close, window=14)
    
    # Estocástico
    stoch_k, stoch_d = stochastic(high, low, close, k_window=14)
    
    fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
    
    # Preço + Bollinger
    axes[0].plot(sample['timestamp'], close, label='Close', linewidth=0.8, color='#2c3e50')
    axes[0].plot(sample['timestamp'], bb_upper, label='Upper BB (2σ)', linewidth=0.8, color='#e74c3c', linestyle='--')
    axes[0].plot(sample['timestamp'], bb_mid, label='Middle BB (SMA20)', linewidth=0.8, color='#f39c12')
    axes[0].plot(sample['timestamp'], bb_lower, label='Lower BB (2σ)', linewidth=0.8, color='#27ae60', linestyle='--')
    axes[0].fill_between(sample['timestamp'], bb_upper, bb_lower, alpha=0.05, color='gray')
    axes[0].set_title('Bandas de Bollinger (20, 2σ)', fontweight='bold')
    axes[0].legend(fontsize=8)
    axes[0].spines['top'].set_visible(False)
    axes[0].spines['right'].set_visible(False)
    
    # RSI
    axes[1].plot(sample['timestamp'], rsi_14, color='#8e44ad', linewidth=0.9)
    axes[1].axhline(70, color='#e74c3c', linewidth=1, linestyle='--', label='Sobrecomprado (70)')
    axes[1].axhline(30, color='#27ae60', linewidth=1, linestyle='--', label='Sobrevendido (30)')
    axes[1].axhline(50, color='gray', linewidth=0.6, linestyle=':')
    axes[1].fill_between(sample['timestamp'], rsi_14, 70, where=rsi_14>=70, alpha=0.2, color='#e74c3c')
    axes[1].fill_between(sample['timestamp'], rsi_14, 30, where=rsi_14<=30, alpha=0.2, color='#27ae60')
    axes[1].set_title('RSI (14)', fontweight='bold')
    axes[1].set_ylim(0, 100)
    axes[1].legend(fontsize=8)
    axes[1].spines['top'].set_visible(False)
    axes[1].spines['right'].set_visible(False)
    
    # Estocástico
    axes[2].plot(sample['timestamp'], stoch_k, color='#2980b9', linewidth=0.9, label='%K')
    axes[2].plot(sample['timestamp'], stoch_d, color='#e67e22', linewidth=0.9, label='%D (3)')
    axes[2].axhline(80, color='#e74c3c', linewidth=1, linestyle='--', label='Overbought (80)')
    axes[2].axhline(20, color='#27ae60', linewidth=1, linestyle='--', label='Oversold (20)')
    axes[2].set_title('Estocástico (14, 3)', fontweight='bold')
    axes[2].set_ylim(0, 100)
    axes[2].set_xlabel('Data')
    axes[2].legend(fontsize=8)
    axes[2].spines['top'].set_visible(False)
    axes[2].spines['right'].set_visible(False)
    
    fig.tight_layout()
    plt.show()

## 4. Bandas de Regressão

In [ ]:
if 'df' in dir() and not df.empty:
    sample = df.tail(500).copy().reset_index(drop=True)
    log_price = np.log(sample['close'])
    
    reg_upper, reg_fitted, reg_lower = regression_bands(log_price, window=100, k=2.0)
    
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(sample['timestamp'], log_price, label='Log-preço', linewidth=0.8, color='#2c3e50')
    ax.plot(sample['timestamp'], reg_fitted, label='Regressão (fitted)', linewidth=1.2, color='#f39c12')
    ax.plot(sample['timestamp'], reg_upper, label='Upper (2σ resid)', linewidth=0.8, color='#e74c3c', linestyle='--')
    ax.plot(sample['timestamp'], reg_lower, label='Lower (2σ resid)', linewidth=0.8, color='#27ae60', linestyle='--')
    ax.fill_between(sample['timestamp'], reg_upper, reg_lower, alpha=0.05, color='gray')
    ax.set_title('Bandas de Regressão Rolling (janela=100, k=2.0)', fontweight='bold')
    ax.set_ylabel('Log-preço')
    ax.set_xlabel('Data')
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    fig.tight_layout()
    plt.show()

## 5. Score de Confluência Técnica

In [ ]:
if 'df' in dir() and not df.empty:
    sample = df.tail(1000).copy().reset_index(drop=True)
    
    score = technical_confluence_score(
        sample['close'], sample['high'], sample['low']
    )
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    
    ax1.plot(sample['timestamp'], sample['close'], linewidth=0.8, color='#2c3e50')
    ax1.set_title('Preço de Fechamento', fontweight='bold')
    ax1.set_ylabel('Preço')
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    
    colors = ['#27ae60' if v > 0 else '#e74c3c' if v < 0 else '#95a5a6' for v in score]
    ax2.bar(sample['timestamp'], score, color=colors, alpha=0.7, width=0.001)
    ax2.axhline(2, color='#27ae60', linewidth=1, linestyle='--', label='Threshold long (2)')
    ax2.axhline(-2, color='#e74c3c', linewidth=1, linestyle='--', label='Threshold short (-2)')
    ax2.axhline(0, color='black', linewidth=0.5)
    ax2.set_title('Technical Confluence Score (não é probabilidade)', fontweight='bold')
    ax2.set_ylabel('Score (-3 a +3)')
    ax2.set_ylim(-3.5, 3.5)
    ax2.legend(fontsize=9)
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    
    fig.tight_layout()
    plt.show()
    
    print('Distribuição do score:')
    print(score.value_counts().sort_index())

## 6. Estatísticas descritivas dos retornos

In [ ]:
if 'df' in dir() and not df.empty:
    log_ret = log_return(df['close']).dropna()
    
    print('=== Estatísticas dos Retornos Logarítmicos ===')
    print(log_ret.describe())
    print(f'\nSkewness: {log_ret.skew():.4f}')
    print(f'Kurtosis: {log_ret.kurt():.4f}')
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].hist(log_ret, bins=200, color='#3498db', alpha=0.7, edgecolor='none')
    axes[0].axvline(log_ret.mean(), color='red', linewidth=1.5, label=f'Média: {log_ret.mean():.5f}')
    axes[0].axvline(0, color='black', linewidth=1, linestyle='--')
    axes[0].set_title(f'Distribuição de Retornos — {symbol} {interval}', fontweight='bold')
    axes[0].set_xlabel('Retorno Log')
    axes[0].set_ylabel('Frequência')
    axes[0].legend()
    axes[0].spines['top'].set_visible(False)
    axes[0].spines['right'].set_visible(False)
    
    # QQ-plot manual
    from scipy import stats
    (osm, osr), (slope, intercept, r) = stats.probplot(log_ret.values)
    axes[1].scatter(osm, osr, s=1, alpha=0.3, color='#3498db')
    x_line = np.array([osm[0], osm[-1]])
    axes[1].plot(x_line, slope * x_line + intercept, 'r-', linewidth=1.5)
    axes[1].set_title('QQ-Plot (Normal)', fontweight='bold')
    axes[1].set_xlabel('Quantis Teóricos')
    axes[1].set_ylabel('Quantis Observados')
    axes[1].spines['top'].set_visible(False)
    axes[1].spines['right'].set_visible(False)
    
    fig.tight_layout()
    plt.show()